In [8]:
import numpy as np
import trimesh
import qoi
import os
import json
import struct

In [9]:
scene = trimesh.load('/home/j/Documents/code/vendor/glTF-Sample-Assets/Models/Sponza/glTF/Sponza.gltf')

In [10]:
models = []
model_vertices = []
model_indices = []
textures = {}

In [11]:
for node_name in scene.graph.nodes_geometry:
    mesh_name = scene.graph[node_name][1]
    mesh = scene.geometry[mesh_name]
    
    transform = scene.graph[node_name][0]
    mesh = mesh.copy()
    mesh.apply_transform(transform)
    
    positions = np.array(mesh.vertices)
    normals = np.array(mesh.vertex_normals)
    texcoords = np.array(mesh.visual.uv)
    indices = np.array(mesh.faces)
    assert positions.shape[0] == normals.shape[0] == texcoords.shape[0], "Mismatch in vertex attributes"

    pixels = np.asarray(mesh.visual.material.baseColorTexture)
    if pixels.shape[2] == 4:
        average_color = pixels[pixels[:, :, 3] > 0][:, :3].mean(axis=(0))
    else:
        average_color = pixels.mean(axis=(0, 1))

    model = {
        'first_vertex': len(model_vertices),
        'first_index': len(model_indices),
        'num_vertices': len(positions),
        'num_indices': len(indices) * 3,
        'diffuse': [int(x) for x in average_color]
    }
    for position, normal in zip(positions, normals):
        model_vertices.append({
            'position': position,
            'normal': normal,
        })
    for index in indices:
        model_indices.extend(index)
    models.append(model)


In [12]:
!mkdir -p data/models/sponza/textures

with open('data/models/sponza/models.json', 'w') as f:
    json.dump(models, f, indent=4)

with open('data/models/sponza/vertices.bin', 'wb') as f:
    f.write(struct.pack('<I', len(model_vertices)))
    for vertex in model_vertices:
        f.write(struct.pack('<ffff', *vertex['position'], 0.0))
        f.write(struct.pack('<ffff', *vertex['normal'], 0.0))

assert len(model_indices) % 3 == 0
with open('data/models/sponza/indices.bin', 'wb') as f:
    f.write(struct.pack('<I', len(model_indices)))
    for index in model_indices:
        f.write(struct.pack('<I', index))

In [13]:
display(len(model_vertices))
display(len(model_indices))

192496

786801